# Evaluating the tutor: a data scientist's walkthrough

*For learning. Nothing here runs in CI, and nothing costs money: it replays recordings of real runs.*

## What's going on, in one paragraph

The tutor in "Ask about this lesson" now has **prepared answers**: 1,728 answers to its standard
questions, written ahead of time by a strong AI model, so Ask works instantly even when no AI is running.
A second AI model (the **judge**) checked every answer before it went live. Now we'd like a **cheaper**
judge to keep checking answers every day. Before trusting it, we test it, the way a data scientist would
test any measuring tool: is it accurate, is it consistent, how sure are we, and is it worth the money?

## Words you'll meet

| Word | Plain meaning |
|---|---|
| **Judge** | An AI model asked to grade another AI's answer against a checklist |
| **Reference** | The judge we compare against (here Sonnet). Not the truth: just another opinion |
| **Ground truth / golden set** | Answers a **person** has graded. The closest thing to the truth we have |
| **Dev / test / holdout** | Practice set (try things), exam set (check once before a release), locked set (big decisions only) |
| **Stratified sample** | A sample that deliberately includes every group in fair amounts, instead of pure luck |
| **Confidence interval** | The range the true value probably sits in. Narrow means sure; wide means unsure |
| **Kappa / AC1** | How much two judges agree, beyond what luck alone would give |
| **Bootstrap** | Re-running the analysis on many shuffled copies of the data to see how much the result wobbles |
| **p-value** | How surprising a result would be if nothing were really going on. Small means "probably real" |

## How to use this notebook

Run the cells from top to bottom (Shift + Enter). Cells that start with `# TODO` are **exercises**: write
the function, then run the test cell below it. Green means right. The answers are in
`evaluating_the_tutor_answers.py` beside this notebook, if you get stuck.

In [ ]:
import json
import math
import random
from collections import Counter
from pathlib import Path

import ipytest
import pandas as pd
import pytest

from drill import tutor_prepared as tp
from drill import tutor_prepared_eval as tpe
from drill.content.build import build_data

ipytest.autoconfig(clean="test_*")

DATA = build_data()                      # every lesson, as the page sees it
ANSWERS = tp.load()                      # the prepared answers: {lesson_id: {question key: record}}
CHIPS = tp.chips(DATA)                   # the standard questions, as the page asks them
RECORDINGS = Path(tp.__file__).parent / "content" / "prepared_eval"
REC = json.loads((RECORDINGS / "dev_sample_haiku.json").read_text())
PHASE = tpe.phases(DATA)                 # each lesson's phase (or the platform track)

print(f"{sum(len(v) for v in CHIPS.values())} questions across {len(CHIPS)} lessons")
print(f"{sum(len(v) for v in ANSWERS.values())} prepared answers; status: {Counter(r['status'] for v in ANSWERS.values() for r in v.values())}")
print(f"recorded judge run: {REC['params']['judge_model']} on {len(REC['answers'])} answers")

**What that cell did.** It loaded four things: every lesson (`DATA`), every prepared answer (`ANSWERS`),
the list of standard questions (`CHIPS`), and a **recording** (`REC`) of the real run where Haiku judged a
sample of answers. Everything below works on these.

## 0. Write the plan before looking at results

> **In plain words.** Decide what "good enough" means *before* you see the numbers. Otherwise it's too
> easy to move the goalposts after a disappointing result. The course calls this pre-registration
> (lesson "Write the plan before the data").

Honest note: the first Haiku run happened before this plan was written. So treat that run as
**exploration**, and this plan as the rule for the **next** runs (a new judge prompt, or a new judge).

In [ ]:
PLAN = {
    "question": "Can a cheaper judge stand in for careful checking of the tutor's prepared answers, every day?",
    "ground_truth": "a person's verdict on a golden set (section 11); until then, Sonnet as a stand-in, said out loud",
    "primary_metric": "agreement with the ground truth on answers that should FAIL (recall) and on answers that should PASS (specificity)",
    "decision_rule": "trust the judge for daily monitoring only if recall >= 0.8 AND specificity >= 0.8, "
                     "each with the lower end of its lesson-level bootstrap interval >= 0.7",
    "secondary": ["Gwet's AC1 and Cohen's kappa", "cost per answer", "consistency when run twice"],
    "comparisons": "a new judge prompt is compared with McNemar on the same answers; many group-level tests are Holm-adjusted",
    "splits": "tune on dev (or its sample); report on test once; never look at holdout",
}
for k, v in PLAN.items():
    print(f"{k:15} {v}")

## 1. One prepared answer, start to finish

> **In plain words.** Before measuring anything, look at one real example. Every answer is stored with
> the question, the answer, which model wrote it, what the judge said, and a **fingerprint**: a short code
> made from the lesson's text. If the lesson changes, the fingerprint stops matching and the answer is
> hidden until it's rewritten.

In [ ]:
lesson, key = "use_case_score", "learn:term"
chip = next(c for c in CHIPS[lesson] if c["key"] == key)
rec = ANSWERS[lesson][key]

print("Question:", chip["q"])
print()
print("Answer (first 400 characters):")
print(rec["a"][:400], "...")
print()
print("Written by:", rec["model"], "| status:", rec["status"])
print("Sonnet's checklist:", {k: rec["judge"][k] for k in ("correct", "answers_it", "plain", "no_giveaway", "standalone")})
print("Fingerprint still matches the lesson:", rec["fingerprint"] == tp.fingerprint(DATA, lesson, chip))

## 2. Splitting fairly: whole lessons, dealt within each phase

> **In plain words.** If you tune a judge on some answers and then test it on the *same* answers, the test
> is meaningless, like marking your own homework with the answer sheet open. So answers are split into
> practice (**dev**), exam (**test**) and locked (**holdout**) groups.
>
> Two rules from the course:
> - **Keep related things together.** All of a lesson's questions share its text, so a whole lesson goes
>   into one group ("the leakage rule").
> - **Deal fairly.** Make sure every phase lands in every group, like dealing cards to each player.

The first attempt split by a random-looking code from each lesson's id. Look what happened:

In [ ]:
plain = {lid: ("dev" if tpe._bucket(lid) < 60 else "test" if tpe._bucket(lid) < 80 else "holdout") for lid in CHIPS}
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in plain], name="phase"), pd.Series(list(plain.values()), name="split"))
table[["dev", "test", "holdout"]]

**What to look for:** the zeros. Design and Ship got **no** holdout lessons, so you could never check those
phases at the end. Pure chance missed small groups, exactly what the lesson "Dev, test and holdout sets"
warns about.

**Exercise.** Deal each phase's lessons in turn: dev, dev, dev, test, holdout, then start again. Keep a
separate count **for each phase**.

In [ ]:
PATTERN = ["dev", "dev", "dev", "test", "holdout"]  # 3 dev, 1 test, 1 holdout: 60/20/20

In [ ]:
def deal_by_phase(lessons_by_phase):
    """{phase: [lesson ids]} -> {lesson id: split}, each phase dealt in the pattern above."""
    # TODO: for each phase, give its lessons PATTERN[0], PATTERN[1], ... in turn, starting again after 5
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_every_phase_with_five_lessons_reaches_every_split():
    dealt = deal_by_phase({"Frame": ["f1", "f2", "f3", "f4", "f5"], "Ship": ["s1", "s2", "s3", "s4", "s5", "s6"]})
    assert {dealt[l] for l in ["f1", "f2", "f3", "f4", "f5"]} == {"dev", "test", "holdout"}
    assert {dealt[l] for l in ["s1", "s2", "s3", "s4", "s5"]} == {"dev", "test", "holdout"}


def test_each_phase_starts_its_own_pattern():
    dealt = deal_by_phase({"A": ["a1"], "B": ["b1", "b2", "b3", "b4"]})
    assert dealt["a1"] == "dev" and dealt["b4"] == "test"


def test_the_pattern_repeats():
    dealt = deal_by_phase({"A": [f"a{i}" for i in range(1, 8)]})
    assert [dealt[f"a{i}"] for i in range(1, 8)] == ["dev", "dev", "dev", "test", "holdout", "dev", "dev"]

The project then **freezes** the result in a file (`prepared_splits.json`), so adding lessons later never
moves old ones between groups (lesson "Splits that don't move"). Every phase now reaches every group:

In [ ]:
frozen = tpe.load_splits()
table = pd.crosstab(pd.Series([PHASE.get(l, "") for l in frozen], name="phase"), pd.Series(list(frozen.values()), name="split"))
table[["dev", "test", "holdout"]]

## 3. The sample, and what the two judges said

> **In plain words.** Judging all 1,025 practice answers takes an hour, so we judged a **sample** of 301.
> Failures are rare (Sonnet failed about 2.5%), and a purely random sample would hold only about 8 of them,
> too few to learn anything. So the sample took **every** failure, then a fair spread of the rest across
> phases and question types. That's a **stratified sample**.

Below, every judged answer becomes one row of a table: its lesson, phase, question type, length, and the
two judges' verdicts. **Sonnet** is the reference; **Haiku** is the cheaper judge we're testing.

In [ ]:
df = pd.DataFrame(REC["answers"])
df["phase"] = df.lesson_id.map(PHASE)
df["kind"] = df.key.map(tpe.kind_of)
df["words"] = [len(ANSWERS[r.lesson_id][r.key]["a"].split()) for r in df.itertuples()]
df = df.rename(columns={"passes": "haiku", "reference_pass": "sonnet"})
df["agree"] = df.haiku == df.sonnet
print(len(df), "answers from", df.lesson_id.nunique(), "lessons")
df[["lesson_id", "kind", "phase", "words", "sonnet", "haiku"]].head(8)

## 4. The confusion matrix: four kinds of outcome

> **In plain words.** For a checker whose job is to catch bad answers, every answer lands in one of four
> boxes:
>
> | | Reference says FAIL | Reference says PASS |
> |---|---|---|
> | **Judge says FAIL** | caught it (true positive) | false alarm |
> | **Judge says PASS** | missed it | correctly let through (true negative) |
>
> Here "positive" means "flagged as a problem". From the boxes:
> - **Recall**: of the truly bad answers, how many did the judge catch? (Missing bad answers is the danger.)
> - **Precision**: of the answers the judge flagged, how many were really bad? (Low means lots of false alarms.)
> - **Specificity**: of the good answers, how many did it let through?

**Exercise.** `judge` and `reference` are lists of True (pass) or False (fail). Count the four boxes, then
work out recall, precision and specificity. If a rate has nothing to divide by, use `None`.

In [ ]:
def confusion(judge, reference):
    """The four boxes and three rates, where 'positive' means the answer FAILED (a problem was flagged)."""
    # TODO: caught = judge failed and reference failed; false_alarm = judge failed, reference passed;
    #       missed = judge passed, reference failed; let_through = both passed.
    #       recall = caught / (caught + missed); precision = caught / (caught + false_alarm);
    #       specificity = let_through / (let_through + false_alarm)
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_the_four_boxes_and_rates():
    judge = [False, False, True, False, True, True]
    ref = [False, True, False, False, True, True]
    out = confusion(judge, ref)
    assert (out["caught"], out["false_alarm"], out["missed"], out["let_through"]) == (2, 1, 1, 2)
    assert out["recall"] == 2 / 3 and out["precision"] == 2 / 3 and out["specificity"] == 2 / 3


def test_nothing_to_divide_by():
    assert confusion([True], [True])["recall"] is None

In [ ]:
cm = confusion(list(df.haiku), list(df.sonnet))
print(pd.DataFrame([[cm["caught"], cm["false_alarm"]], [cm["missed"], cm["let_through"]]],
                   index=["Haiku FAIL", "Haiku PASS"], columns=["Sonnet FAIL", "Sonnet PASS"]))
print(f"\nrecall {cm['recall']:.0%}   precision {cm['precision']:.0%}   specificity {cm['specificity']:.0%}")

**What to look for.** Haiku catches most of what Sonnet failed (recall around 85%), but its **precision
is tiny**: almost everything it flags, Sonnet passed. As a daily monitor it would cry wolf constantly.
Remember though: Sonnet isn't the truth. Section 11 is where a person decides.

## 5. How sure is a rate? Intervals, the simple way

> **In plain words.** "36% of 275" is an estimate. With a different sample you'd get a slightly different
> number. A **confidence interval** is the range the true rate probably sits in. The course's rule of thumb:
> the margin is about **100 divided by the square root of the sample size**, in percentage points (so 300
> answers: about plus or minus 6 points).
>
> The **Wilson interval** (lesson "How sure is that pass rate?") does it properly, and behaves well even for
> small groups or rates near 0% or 100%. In words: start from the observed rate, pull it a little toward the
> middle (more when the sample is small), and spread out by an amount that shrinks as the sample grows.

**Exercise.** Return `(rate, low, high)`. With `n = 0`, return `(0.0, 0.0, 0.0)`.

In [ ]:
def wilson(passed, n, z=1.96):
    """A rate and its Wilson 95% interval."""
    # TODO: p = passed / n; centre = (p + z*z/(2n)) / (1 + z*z/n);
    #       half = z * sqrt(p(1-p)/n + z*z/(4n*n)) / (1 + z*z/n); return p, centre - half, centre + half
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_out_of_a_hundred():
    rate, low, high = wilson(90, 100)
    assert rate == 0.9 and round(low, 3) == 0.826 and round(high, 3) == 0.945


def test_nothing_measured():
    assert wilson(0, 0) == (0.0, 0.0, 0.0)


def test_it_never_leaves_zero_to_one():
    assert wilson(10, 10)[2] <= 1.0 and wilson(0, 10)[1] >= 0.0

## 6. Answers aren't independent: the lesson-level bootstrap

> **In plain words.** Wilson assumes every answer is a separate, unrelated coin flip. They aren't: answers
> about the **same lesson** tend to pass or fail together (same text, same tricky terms). 301 answers from
> about 100 lessons carry less information than 301 unrelated answers, so Wilson's interval is **too
> narrow**: it makes us sound surer than we are.
>
> The fix is a **bootstrap that resamples whole lessons**: build a new sample by picking lessons at random
> (some twice, some not at all), keeping all of each picked lesson's answers, and recompute the number.
> Do that a thousand times; the middle 95% of the results is the interval. It's a "cluster bootstrap".

This one is written for you; read it line by line.

In [ ]:
def cluster_bootstrap(rows, stat, n_boot=1000, seed=7):
    """95% interval for stat(rows), resampling whole lessons. rows: a DataFrame with a lesson_id column."""
    rng = random.Random(seed)                              # a fixed seed: the same result every time you run it
    by_lesson = {lid: grp for lid, grp in rows.groupby("lesson_id")}
    lessons = list(by_lesson)
    results = []
    for _ in range(n_boot):
        picked = [rng.choice(lessons) for _ in lessons]    # as many lessons as before, chosen with replacement
        sample = pd.concat([by_lesson[l] for l in picked])
        value = stat(sample)
        if value is not None:
            results.append(value)
    results.sort()
    return results[int(0.025 * len(results))], results[int(0.975 * len(results)) - 1]


keep = lambda d: (d[d.sonnet].haiku).mean()                # of Sonnet's passes, the share Haiku passed too
catch = lambda d: (~d[~d.sonnet].haiku).mean() if (~d.sonnet).any() else None

for name, fn, hit, n in (("keep", keep, int(df[df.sonnet].haiku.sum()), int(df.sonnet.sum())),
                         ("catch", catch, int((~df[~df.sonnet].haiku).sum()), int((~df.sonnet).sum()))):
    _, wl, wh = tpe.wilson(hit, n)
    bl, bh = cluster_bootstrap(df, fn)
    print(f"{name:5}: {hit / n:.0%}   Wilson {wl:.0%}-{wh:.0%} (width {wh - wl:.0%})   lesson bootstrap {bl:.0%}-{bh:.0%} (width {bh - bl:.0%})")

**What to look for.** The lesson-level intervals are **wider**. That's the honest size of our uncertainty.
A professional reports these, not the narrow ones.

**The design effect** puts a number on it: how many times wider (squared) the honest interval is. A design
effect of 2 means our 301 answers are worth about 150 independent ones.

In [ ]:
_, wl, wh = tpe.wilson(int(df[df.sonnet].haiku.sum()), int(df.sonnet.sum()))
bl, bh = cluster_bootstrap(df, keep)
effect = ((bh - bl) / (wh - wl)) ** 2
print(f"design effect about {effect:.1f}: the {int(df.sonnet.sum())} passed answers are worth about {df.sonnet.sum() / effect:.0f} independent ones")

## 7. Do two judges agree? Kappa, and why it misbehaves here

> **In plain words.** If almost every answer passes, two judges will agree most of the time just by both
> saying "pass". **Cohen's kappa** asks: how much better than luck? 1 is perfect, 0 is no better than
> luck.
>
> - **Observed**: how often they agree.
> - **Chance**: how often they'd agree by luck = (A's pass share x B's pass share) + (A's fail share x B's fail share).
> - **Kappa** = (observed - chance) / (1 - chance). If chance is 1, kappa is undefined: return `None`.

**Exercise.**

In [ ]:
def kappa(a, b):
    """Cohen's kappa between two judges' pass/fail verdicts."""
    # TODO: observed agreement, chance agreement, then (observed - chance) / (1 - chance); None if chance is 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ninety_six_percent_agreement_is_not_as_good_as_it_sounds():
    a = [True] * 96 + [False] * 4
    b = [True] * 98 + [False] * 2
    assert round(kappa(a, b), 2) == 0.66


def test_perfect_agreement():
    assert kappa([True, False, True, False], [True, False, True, False]) == 1.0


def test_no_variation_means_no_answer():
    assert kappa([True] * 5, [True] * 5) is None

> **The kappa paradox.** When one outcome is very rare (here, failing), kappa can come out **low even when
> two judges mostly agree**, because the "chance" part becomes huge. **Gwet's AC1** fixes that by
> estimating chance differently: if `q` is the average pass share of the two judges, chance =
> 2 x q x (1 - q). Then AC1 = (observed - chance) / (1 - chance), the same shape as kappa.

**Exercise.** Write AC1, then compare the two on the example from the test above.

In [ ]:
def ac1(a, b):
    """Gwet's AC1 between two judges' pass/fail verdicts."""
    # TODO: observed agreement; q = (A's pass share + B's pass share) / 2; chance = 2q(1-q);
    #       (observed - chance) / (1 - chance); None if chance is 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_ac1_stays_high_when_judges_mostly_agree_on_rare_failures():
    a = [True] * 96 + [False] * 4
    b = [True] * 98 + [False] * 2
    assert ac1(a, b) > 0.9 > kappa(a, b)


def test_ac1_perfect():
    assert ac1([True, False, True], [True, False, True]) == 1.0

In [ ]:
a, b = list(df.haiku), list(df.sonnet)
print(f"observed agreement {sum(x == y for x, y in zip(a, b)) / len(a):.0%}")
print(f"kappa {kappa(a, b):.2f}   AC1 {ac1(a, b):.2f}")
kl, kh = cluster_bootstrap(df, lambda d: kappa(list(d.haiku), list(d.sonnet)))
al, ah = cluster_bootstrap(df, lambda d: ac1(list(d.haiku), list(d.sonnet)))
print(f"lesson-bootstrap intervals: kappa {kl:.2f} to {kh:.2f}, AC1 {al:.2f} to {ah:.2f}")

**What to look for.** Here both are low, and AC1 doesn't rescue Haiku: the judges genuinely disagree a lot,
it isn't just the paradox. Note too that this sample over-represents failures on purpose, so these numbers
describe the sample. Section 8 fixes that.

## 8. Undoing the over-sampling: weights

> **In plain words.** We put *all* 26 failures in the sample but only about 1 in 4 of the passes. So each
> sampled pass "stands for" about 4 real ones, and each failure stands for 1. Giving every row that
> **weight** turns sample numbers back into estimates for the whole dev set.

In [ ]:
share_failed = REC["metrics"]["population_failed_share"]  # the share of ALL dev answers Sonnet failed (recorded by the run)
w_fail = share_failed / (~df.sonnet).mean()           # how many real answers each sampled failure stands for (relative)
w_pass = (1 - share_failed) / df.sonnet.mean()
df["weight"] = df.sonnet.map({False: w_fail, True: w_pass})


def weighted_agreement(d):
    return (d.agree * d.weight).sum() / d.weight.sum()


lo, hi = cluster_bootstrap(df, weighted_agreement)
print(f"failures are {share_failed:.1%} of dev but {(~df.sonnet).mean():.1%} of the sample")
print(f"agreement in the sample {df.agree.mean():.0%}; weighted to dev's real mix {weighted_agreement(df):.0%} (lesson bootstrap {lo:.0%}-{hi:.0%})")

## 9. Where do they disagree, and why?

> **In plain words.** One overall number hides *where* things go wrong. Split by group. But testing many
> groups creates a new trap: test 13 groups and one will look "significant" by luck alone. So adjust the
> p-values for how many you tested.

First, agreement by phase and by question type, each with a p-value for "is this group's keep rate
different from the rest?"

In [ ]:
def two_rate_p(hit1, n1, hit2, n2):
    """A simple two-sided test that two rates differ (normal approximation): small p means probably real."""
    if not n1 or not n2:
        return 1.0
    p = (hit1 + hit2) / (n1 + n2)
    se = math.sqrt(p * (1 - p) * (1 / n1 + 1 / n2)) or 1e-9
    z = abs(hit1 / n1 - hit2 / n2) / se
    return math.erfc(z / math.sqrt(2))


rows = []
passed = df[df.sonnet]
for by in ("phase", "kind"):
    for group, g in passed.groupby(by):
        rest = passed[passed[by] != group]
        rows.append({"by": by, "group": group, "n": len(g), "keep_rate": round(g.haiku.mean(), 2),
                     "p": two_rate_p(int(g.haiku.sum()), len(g), int(rest.haiku.sum()), len(rest))})
groups = pd.DataFrame(rows)
groups

**Exercise: the Holm adjustment.** Sort the p-values from smallest to largest. Multiply the smallest by the
number of tests, the next by one fewer, and so on. Each adjusted value must be at least as big as the one
before it, and never more than 1. Return them **in the original order**.

In [ ]:
def holm(pvalues):
    """Holm-adjusted p-values, in the same order as given."""
    # TODO: order = indexes sorted by p; walk them, adjusted = max(previous, min(1, (m - rank) * p))
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_holm_on_three():
    assert [round(x, 3) for x in holm([0.01, 0.04, 0.03])] == [0.03, 0.06, 0.06]


def test_never_above_one():
    assert holm([0.5, 0.9]) == [1.0, 1.0]

In [ ]:
groups["p_adjusted"] = holm(list(groups.p))
groups.sort_values("p_adjusted").round(3)

**What to look for.** Which groups stay below 0.05 *after* adjusting. Those are real differences worth
acting on; the rest may be luck.

Now **why**. Haiku writes a reason for every fail. Sorting reasons into categories by keywords is rough,
and a person reading them would do better, but it shows the shape:

In [ ]:
CATEGORIES = {
    "unexplained jargon": ("jargon", "unexplained", "never explain", "without explain", "not explained", "acronym", "term"),
    "claims the lesson doesn't make": ("lesson never", "lesson does not", "lesson doesn't", "not in the lesson", "overstat", "overclaim", "contradict", "goes beyond"),
    "too long": ("words", "too long", "sentences", "length"),
    "gives something away": ("exercise", "solution", "gives away", "answer away"),
}


def category(note):
    note = (note or "").lower()
    return next((name for name, words in CATEGORIES.items() if any(w in note for w in words)), "other")


flagged = df[~df.haiku & df.sonnet].copy()
flagged["category"] = flagged.notes.map(category)
print(flagged.category.value_counts().to_string())
for name in flagged.category.unique():
    print(f"\n{name}, for example:")
    for note in flagged[flagged.category == name].notes.head(2):
        print("  -", note[:150])

## 10. Is the judge biased by length?

> **In plain words.** AI judges are known to be swayed by things that shouldn't matter, length most of all.
> If Haiku fails long answers more often, part of its strictness might just be about length. Split answers
> into four equal-sized length groups and compare pass rates.

In [ ]:
df["length_group"] = pd.qcut(df.words, 4, labels=["shortest", "short", "long", "longest"])
by_length = df.groupby("length_group", observed=True).agg(answers=("words", "size"), median_words=("words", "median"),
                                                         sonnet_passes=("sonnet", "mean"), haiku_passes=("haiku", "mean")).round(2)
print(by_length.to_string())
print(f"\ncorrelation of length with passing: Haiku {df.words.corr(df.haiku.astype(float)):+.2f}, Sonnet {df.words.corr(df.sonnet.astype(float)):+.2f}")

**What to look for.** If Haiku's pass rate falls clearly as answers get longer while Sonnet's doesn't,
that's a length bias. A correlation near 0 means length isn't driving it.

## 11. Is the judge consistent? Run it twice

> **In plain words.** A thermometer that reads differently every time you use it is useless, however
> accurate it is on average. Judges are the same: run Haiku twice on the **same** answers and see how often
> it changes its mind. If it flips often, every other number in this notebook is noisy.

In [ ]:
rerun_path = RECORDINGS / "dev_sample_haiku_rerun.json"
if rerun_path.exists():
    rerun = {(a["lesson_id"], a["key"]): a["passes"] for a in json.loads(rerun_path.read_text())["answers"]}
    first = list(df.haiku)
    second = [rerun.get((r.lesson_id, r.key)) for r in df.itertuples()]
    both = [(x, y) for x, y in zip(first, second) if y is not None]
    same = sum(x == y for x, y in both)
    print(f"Haiku gave the same verdict twice on {same} of {len(both)} answers ({same / len(both):.0%})")
    print(f"kappa between its two runs {kappa([x for x, _ in both], [y for _, y in both]):.2f}, AC1 {ac1([x for x, _ in both], [y for _, y in both]):.2f}")
else:
    print("The second run's recording isn't here yet: run `uv run drill tutor prepared-eval --profile DEFAULT --split dev --sample 300` again.")

## 12. Comparing two versions: McNemar, and how many answers you need

> **In plain words.** When you change the judge's instructions and run it again on the **same** answers,
> only the answers where the two runs **disagree** tell you which is better. If there's no real
> difference, each disagreement is a coin flip. McNemar's test asks how surprising your split of
> disagreements would be under coin flips (lesson "Real win or luck").

**Exercise.** The chance of `min(b, c)` or fewer heads in `b + c` fair flips, doubled, capped at 1.
`math.comb(n, k)` counts the ways to get `k` heads in `n` flips; there are `2 ** n` outcomes in all.

In [ ]:
def mcnemar(b, c):
    """Exact two-sided McNemar p-value from the disagreements alone."""
    # TODO: n = b + c; add up math.comb(n, k) for k from 0 to min(b, c); divide by 2 ** n; double; cap at 1
    raise NotImplementedError

In [ ]:
%%ipytest -qq


def test_twelve_against_eight_is_too_close_to_call():
    assert round(mcnemar(12, 8), 3) == 0.503


def test_fifteen_against_two_is_real():
    assert mcnemar(15, 2) < 0.01


def test_it_does_not_matter_which_side_is_which():
    assert mcnemar(3, 9) == mcnemar(9, 3)


def test_no_disagreements():
    assert mcnemar(0, 0) == 1.0

> **How many answers do you need?** Before running an experiment, check it can actually detect the
> difference you care about. Otherwise a "no difference" result just means "too small to tell". This is
> called **power**: the chance the test spots a real difference of a given size.
>
> The cell below simulates it: imagine the new prompt really fixes some share of the old run's mistakes and
> breaks a smaller share of its right answers, run McNemar many times on made-up data like that, and count how
> often it says "real" (p under 0.05).

In [ ]:
def power(n_answers, fixes, breaks, trials=2000, seed=7):
    """Chance McNemar spots the difference when the new version fixes `fixes` and breaks `breaks` (shares of answers)."""
    rng = random.Random(seed)
    hits = 0
    for _ in range(trials):
        b = sum(rng.random() < fixes for _ in range(n_answers))
        c = sum(rng.random() < breaks for _ in range(n_answers))
        hits += mcnemar(b, c) < 0.05
    return hits / trials


for n in (100, 300, 1000):
    print(f"{n:5} answers: power {power(n, fixes=0.06, breaks=0.02):.0%} to detect fixing 6% and breaking 2%")

**What to look for.** Around 80% is the usual target. If 300 answers give much less, the experiment can't
answer its question: use more answers, or only look for bigger effects.

The project has the same McNemar function: `uv run drill tutor prepared-compare RUN_A RUN_B` runs it on two
real runs, answer by answer.

## 13. Is it worth the money? Cost against quality

> **In plain words.** A cheaper judge is only a bargain if it's good enough. Put cost and quality side by
> side (the course's lesson on choosing a model by quality, cost and speed).

In [ ]:
m = REC["metrics"]
haiku_per_answer = m["judge_cost_per_answer_usd"]
# Sonnet's judging during drafting, from the workspace's usage table on 2026-10-17: 2,094 calls
sonnet_per_answer = tp.cost("databricks-claude-sonnet-5-5", 3_429_008, 2_212_630) / 2094

options = pd.DataFrame([
    {"judge": "Sonnet 5.5 (reference)", "cents_per_answer": 100 * sonnet_per_answer, "agrees_with_reference": 1.0,
     "precision_vs_reference": 1.0, "note": "the reference: agreement with itself is 1 by definition"},
    {"judge": "Haiku 5.5", "cents_per_answer": 100 * haiku_per_answer, "agrees_with_reference": round(weighted_agreement(df), 2),
     "precision_vs_reference": round(cm["precision"], 2), "note": f"{int(m['judge_rate_limited'])} calls rate-limited"},
])
options["cost_for_1000_answers_usd"] = (options.cents_per_answer * 10).round(2)
options.round(2)

**What to look for.** Haiku is cheaper per answer, but at this agreement and precision it would bury real
problems under false alarms. The honest comparison, though, needs a person's verdicts as the truth,
which is the next section.

## 14. Your turn: build the golden set (the ground truth)

> **In plain words.** Everything so far compared one AI with another. To know which is right, a **person**
> grades a sample: that's the **golden set**. A professional would also have **two people** grade the same
> answers separately and measure how much *they* agree. That's the ceiling: no judge can be expected to
> beat human-to-human agreement.

The cell below writes a spreadsheet of about 60 answers to grade: every disagreement Haiku and Sonnet had
on failures, plus a spread of the rest. Open it, fill the `your_verdict` column with `pass` or `fail` (and
`second_person` if someone else grades too), save it, then run the cell after.

In [ ]:
golden_path = Path("golden_to_label.csv")
if not golden_path.exists():
    on_failures = df[~df.sonnet | ~df.haiku]
    pick = pd.concat([on_failures[~on_failures.sonnet], on_failures[on_failures.sonnet].sample(n=min(25, int(on_failures.sonnet.sum())), random_state=7),
                      df[df.sonnet & df.haiku].sample(n=10, random_state=7)]).drop_duplicates(["lesson_id", "key"])
    sheet = pick[["lesson_id", "key"]].copy()
    sheet["question"] = [ANSWERS[r.lesson_id][r.key]["q"] for r in pick.itertuples()]
    sheet["answer"] = [ANSWERS[r.lesson_id][r.key]["a"] for r in pick.itertuples()]
    sheet["your_verdict"] = ""
    sheet["second_person"] = ""
    sheet.sample(frac=1, random_state=7).to_csv(golden_path, index=False)  # shuffled, and judges' verdicts hidden: no peeking
    print(f"wrote {len(sheet)} answers to {golden_path.resolve()}")
else:
    print(f"{golden_path} already exists: fill it in, then run the next cell")

In [ ]:
labels = pd.read_csv(golden_path).fillna("")
labels = labels[labels.your_verdict.str.strip().str.lower().isin(["pass", "fail"])]
if len(labels):
    merged = labels.merge(df[["lesson_id", "key", "haiku", "sonnet"]], on=["lesson_id", "key"])
    you = merged.your_verdict.str.strip().str.lower() == "pass"
    for judge in ("haiku", "sonnet"):
        c = confusion(list(merged[judge]), list(you))
        print(f"{judge:6} vs you: recall {c['recall']}, precision {c['precision']}, specificity {c['specificity']}, "
              f"AC1 {ac1(list(merged[judge]), list(you)):.2f}")
    two = merged[merged.second_person.str.strip().str.lower().isin(["pass", "fail"])]
    if len(two):
        other = two.second_person.str.strip().str.lower() == "pass"
        mine = two.your_verdict.str.strip().str.lower() == "pass"
        print(f"you vs the second person on {len(two)}: kappa {kappa(list(mine), list(other))}, AC1 {ac1(list(mine), list(other)):.2f}  <- the ceiling")
    print("\nWhen you're happy: record them with `uv run drill tutor prepared --check lesson_id/key` or `--reject lesson_id/key`")
else:
    print("No labels yet: fill in your_verdict in golden_to_label.csv")

## 15. The experiment that really matters: does it help learners?

> **In plain words.** A judge is only a stand-in. The real question is whether answers **help people
> learn**. The app already records what tells us: **Got it / Still stuck** on each answer, and whether the
> learner got the **next question right**. So the real experiment runs on live use.

A professional plan for it, written before any data:

| Part | Plan |
|---|---|
| **Question** | Do prepared answers help as much as live answers from the AI? |
| **Groups** | Each learner is randomly assigned, once, to see prepared answers or live ones for the standard questions (the same person always sees the same kind, so they don't compare) |
| **Primary metric** | Share of first answers that get **Got it** |
| **Secondary** | Next question right first time; Still stuck rate; cost per answer |
| **Guardrail** | Stop early if Still stuck rises by more than 10 points |
| **Analysis** | Compare the two groups' rates with intervals; cluster by learner, since one person asks many questions |
| **Decision rule** | Keep prepared answers if Got it is no more than 3 points worse, since they're instant and free |

How many questions would it take? A common rule of thumb (Lehr's rule) for comparing two rates: about
16 x rate x (1 - rate) / difference squared **per group**.

In [ ]:
for got_it, diff in ((0.7, 0.05), (0.7, 0.03), (0.5, 0.05)):
    per_group = 16 * got_it * (1 - got_it) / diff ** 2
    print(f"Got it around {got_it:.0%}, to detect a {diff:.0%} difference: about {per_group:,.0f} answered questions per group")

**What to look for.** Small differences need a lot of traffic. For a learning app with a few users that
means running for a long time, or only looking for bigger differences. That trade-off is the honest
answer, and it's why the cheaper checks above (judges, golden sets) are still worth having.

## What you used

| Idea | Course lesson | In this notebook |
|---|---|---|
| Write the plan first | Write the plan before the data | section 0 |
| Fair splits, stratify, golden set | Dev, test and holdout sets | sections 2, 3, 14 |
| Splits that never move | Splits that don't move | section 2 |
| How many examples | How many examples do you need? | sections 5, 12, 15 |
| Pass rate with its margin | How sure is that pass rate? | section 5 |
| Agreement beyond chance | Do two judges agree? | section 7 |
| Real win or luck | Real win or luck | section 12 |
| Must-pass gate | Must-pass gate | section 0's decision rule |
| Quality, cost and speed together | Choosing a model | section 13 |

Beyond the course: the confusion matrix, the lesson-level bootstrap and design effect, Gwet's AC1, sample
weights, the Holm adjustment, length bias, test-retest consistency, power by simulation, and an online
experiment plan.